# 1. Ekstraksi Data Kualitas Udara — Kabupaten Lamongan

**Mata Kuliah:** Proyek Sains Data — Semester 5
**Sumber Data:** Copernicus Data Space Ecosystem (Sentinel-5P L2)
**Wilayah:** Kabupaten Lamongan, Jawa Timur
**Rentang Waktu:** 24 Agustus 2025 — 24 Agustus 2026
**Polutan:** NO₂, CO, SO₂, CH₄

## Tujuan Notebook
Notebook ini melakukan **tahap ekstraksi data** dalam alur kerja Data Science:
1. Terhubung ke backend **openEO Copernicus Data Space** menggunakan otentikasi *device code flow*.
2. Menentukan area kajian (bounding box/polygon) Kabupaten Lamongan.
3. Menarik data satelit **Sentinel-5P L2** untuk 4 polutan (NO₂, CO, SO₂, CH₄).
4. Melakukan agregasi temporal harian (mean) dan agregasi spasial berdasarkan polygon wilayah.
5. Mengekspor hasil ke format **NetCDF (.nc)** lalu mengonversinya ke **CSV**.

> **Catatan:** Notebook ini diasumsikan berada dalam folder `notebooks/`, sehingga path data
> relatif menggunakan `../data/nc/` dan `../data/csv/` (satu tingkat di atas folder notebook).

## 2. Import Pustaka

- `openeo` — client Python untuk mengakses backend openEO Copernicus Data Space.
- `xarray` — membaca dan memanipulasi data NetCDF multidimensi (waktu x lat x lon).
- `pandas` — konversi data menjadi tabel dan ekspor ke CSV.
- `os` — membuat folder output jika belum ada.

In [1]:
import os
import openeo
import xarray as xr
import pandas as pd

# Pastikan folder output tersedia sebelum proses ekstraksi dimulai
NC_DIR = "../data/nc/"
CSV_DIR = "../data/csv/"
os.makedirs(NC_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Folder output siap:")
print(" -", os.path.abspath(NC_DIR))
print(" -", os.path.abspath(CSV_DIR))

Folder output siap:
 - d:\Semester 5\Proyek sain data\PSD\data\nc
 - d:\Semester 5\Proyek sain data\PSD\data\csv


## 3. Koneksi & Otentikasi ke Copernicus Data Space

Kita terhubung ke backend `openeo.dataspace.copernicus.eu` lalu melakukan otentikasi
menggunakan **OIDC Device Code Flow**. Pada metode ini:

1. Kode akan tercetak di output sel (mis. `https://identity.dataspace.copernicus.eu/...` beserta *user code*).
2. Buka link tersebut di browser, masukkan *user code*, lalu login menggunakan akun
   Copernicus Data Space (CDSE) Anda.
3. Setelah berhasil login di browser, sel kode di bawah akan otomatis melanjutkan proses
   (tidak perlu memasukkan password/token apa pun di notebook).

In [2]:
# Membuka koneksi ke backend openEO Copernicus Data Space
connection = openeo.connect("openeo.dataspace.copernicus.eu")

# Melakukan otentikasi dengan device code flow (interaktif via browser)
connection.authenticate_oidc()

print("Berhasil terhubung sebagai:", connection.describe_account())

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=FQEI-PEHF 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.
Berhasil terhubung sebagai: {'info': {'oidc_userinfo': {'email': 'triswanti1395@gmail.com', 'email_verified': True, 'family_name': "Jannatul Ma'wa", 'given_name': 'Triswanti', 'name': "Triswanti Jannatul Ma'wa", 'preferred_username': 'triswanti1395@gmail.com', 'sub': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}}, 'name': "Triswanti Jannatul Ma'wa", 'user_id': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}


## 4. Menentukan Area Kajian (Kabupaten Lamongan)

Area kajian didefinisikan dalam dua bentuk:
- **Bounding box** — digunakan sebagai `spatial_extent` saat memuat koleksi data (`load_collection`),
  agar volume data yang diunduh dari server tidak terlalu besar.
- **Polygon (GeoJSON)** — digunakan pada tahap `aggregate_spatial` agar nilai polutan yang
  dihitung benar-benar mewakili wilayah administratif Kabupaten Lamongan, bukan sekadar
  kotak persegi pembatasnya.

> **Penting:** Koordinat polygon di bawah adalah **poligon sederhana (perkiraan)** yang
> mengikuti bentuk umum wilayah Kabupaten Lamongan. Untuk hasil yang presisi secara
> administratif, sebaiknya ganti `LAMONGAN_POLYGON` dengan geometri resmi dari sumber
> seperti **Batas Administrasi BIG** atau **GADM** (dibaca via `geopandas`, lalu diekspor
   ke `__geo_interface__`).

In [3]:
# Bounding box (west, south, east, north) — perkiraan wilayah Kabupaten Lamongan
LAMONGAN_BBOX = {
    "west": 112.00,
    "south": -7.35,
    "east": 112.60,
    "north": -6.85,
}

# Polygon sederhana (GeoJSON) yang mengikuti garis luar Kabupaten Lamongan (perkiraan).
# Ganti dengan geometri administratif resmi jika tersedia (mis. hasil geopandas.read_file(shapefile)).
LAMONGAN_POLYGON = {
    "type": "Polygon",
    "coordinates": [[
        [112.00, -7.00],
        [112.10, -7.30],
        [112.25, -7.35],
        [112.40, -7.25],
        [112.55, -7.15],
        [112.60, -7.00],
        [112.50, -6.88],
        [112.30, -6.85],
        [112.10, -6.90],
        [112.00, -7.00],
    ]]
}

print("Bounding box Lamongan:", LAMONGAN_BBOX)

Bounding box Lamongan: {'west': 112.0, 'south': -7.35, 'east': 112.6, 'north': -6.85}


## 5. Rentang Waktu & Parameter Polutan

Sentinel-5P L2 di Copernicus Data Space menyediakan band terpisah untuk tiap polutan pada
koleksi `SENTINEL_5P_L2`. Kita definisikan rentang waktu dan daftar polutan yang akan
diekstraksi (masing-masing dengan nama band openEO dan nama file output).

In [4]:
# Rentang waktu ekstraksi data
TEMPORAL_EXTENT = ["2025-08-24", "2026-08-24"]

# Daftar polutan yang akan diekstraksi.
# 'band' = nama band pada koleksi SENTINEL_5P_L2 di openEO Copernicus Data Space
# 'label' = nama pendek yang dipakai untuk penamaan file output
POLLUTANTS = [
    {"band": "NO2", "label": "no2"},
    {"band": "CO",  "label": "co"},
    {"band": "SO2", "label": "so2"},
    {"band": "CH4", "label": "ch4"},
]

for p in POLLUTANTS:
    print(f"- Band: {p['band']:<5} -> label output: {p['label']}")

- Band: NO2   -> label output: no2
- Band: CO    -> label output: co
- Band: SO2   -> label output: so2
- Band: CH4   -> label output: ch4


## 6. Fungsi Ekstraksi Satu Polutan

Fungsi `extract_pollutant()` di bawah ini merangkum seluruh proses openEO untuk **satu**
band polutan:

1. `load_collection()` — memuat koleksi `SENTINEL_5P_L2`, dibatasi oleh `spatial_extent`,
   `temporal_extent`, dan `bands` (hanya band polutan yang dibutuhkan).
2. `aggregate_temporal_period(period="day", reducer="mean")` — mengagregasi data menjadi
   **rata-rata harian**, karena dalam satu hari bisa terdapat lebih dari satu observasi satelit.
3. `aggregate_spatial(geometries=..., reducer="mean")` — mengagregasi nilai piksel di dalam
   polygon Kabupaten Lamongan menjadi **satu nilai rata-rata per hari** yang mewakili
   seluruh wilayah.
4. `download()` — mengunduh hasil akhir sebagai file **NetCDF (.nc)** ke folder `../data/nc/`.

Contoh di bawah menunjukkan alur lengkap untuk **NO₂**, kemudian di Bagian 7 kita
melakukan *looping* agar alur yang sama berjalan otomatis untuk CO, SO₂, dan CH₄.

In [5]:
def extract_pollutant(connection, band, label, bbox, polygon, temporal_extent, output_dir):
    """
    Menarik data satu polutan dari Sentinel-5P L2, agregasi harian (mean),
    agregasi spasial berdasarkan polygon, lalu diekspor sebagai NetCDF.

    Parameters
    ----------
    connection : openeo.Connection
        Koneksi openEO yang sudah terautentikasi.
    band : str
        Nama band polutan pada koleksi SENTINEL_5P_L2 (mis. "NO2").
    label : str
        Nama pendek untuk penamaan file output (mis. "no2").
    bbox : dict
        Bounding box wilayah kajian (west, south, east, north).
    polygon : dict
        Geometry GeoJSON Polygon wilayah kajian untuk aggregate_spatial.
    temporal_extent : list[str]
        Rentang waktu [start_date, end_date].
    output_dir : str
        Folder tujuan penyimpanan file NetCDF.

    Returns
    -------
    str
        Path file NetCDF hasil ekstraksi.
    """
    print(f"[{label.upper()}] Memuat koleksi SENTINEL_5P_L2 (band={band}) ...")

    # 1. Memuat koleksi data, dibatasi wilayah, waktu, dan band yang relevan
    datacube = connection.load_collection(
        "SENTINEL_5P_L2",
        spatial_extent=bbox,
        temporal_extent=temporal_extent,
        bands=[band],
    )

    # 2. Agregasi temporal: rata-rata harian
    daily_cube = datacube.aggregate_temporal_period(period="day", reducer="mean")

    # 3. Agregasi spasial: rata-rata nilai piksel dalam polygon Kabupaten Lamongan
    spatial_result = daily_cube.aggregate_spatial(geometries=polygon, reducer="mean")

    # 4. Unduh hasil sebagai NetCDF
    output_path = os.path.join(output_dir, f"{label}_lamongan.nc")
    print(f"[{label.upper()}] Menjalankan proses di backend & mengunduh ke {output_path} ...")
    spatial_result.download(output_path, format="netCDF")

    print(f"[{label.upper()}] Selesai -> {output_path}\n")
    return output_path

## 7. Eksekusi Ekstraksi untuk Seluruh Polutan

Selanjutnya kita jalankan `extract_pollutant()` secara berulang (*looping*) untuk keempat
polutan yang telah didefinisikan di Bagian 5: **NO₂, CO, SO₂, CH₄**. Setiap iterasi
menghasilkan satu file `.nc` di `../data/nc/`.

Proses ini dibungkus dengan `try/except` agar jika salah satu polutan gagal diproses
(mis. karena timeout server), proses untuk polutan lain tetap dapat berjalan.

In [6]:
nc_paths = {}

for p in POLLUTANTS:
    try:
        nc_path = extract_pollutant(
            connection=connection,
            band=p["band"],
            label=p["label"],
            bbox=LAMONGAN_BBOX,
            polygon=LAMONGAN_POLYGON,
            temporal_extent=TEMPORAL_EXTENT,
            output_dir=NC_DIR,
        )
        nc_paths[p["label"]] = nc_path
    except Exception as e:
        print(f"[{p['label'].upper()}] Gagal diproses: {e}\n")

print("Ringkasan file NetCDF yang berhasil dibuat:")
for label, path in nc_paths.items():
    print(f" - {label}: {path}")

[NO2] Memuat koleksi SENTINEL_5P_L2 (band=NO2) ...
[NO2] Menjalankan proses di backend & mengunduh ke ../data/nc/no2_lamongan.nc ...
[NO2] Selesai -> ../data/nc/no2_lamongan.nc

[CO] Memuat koleksi SENTINEL_5P_L2 (band=CO) ...
[CO] Menjalankan proses di backend & mengunduh ke ../data/nc/co_lamongan.nc ...
[CO] Selesai -> ../data/nc/co_lamongan.nc

[SO2] Memuat koleksi SENTINEL_5P_L2 (band=SO2) ...
[SO2] Menjalankan proses di backend & mengunduh ke ../data/nc/so2_lamongan.nc ...
[SO2] Selesai -> ../data/nc/so2_lamongan.nc

[CH4] Memuat koleksi SENTINEL_5P_L2 (band=CH4) ...
[CH4] Menjalankan proses di backend & mengunduh ke ../data/nc/ch4_lamongan.nc ...
[CH4] Selesai -> ../data/nc/ch4_lamongan.nc

Ringkasan file NetCDF yang berhasil dibuat:
 - no2: ../data/nc/no2_lamongan.nc
 - co: ../data/nc/co_lamongan.nc
 - so2: ../data/nc/so2_lamongan.nc
 - ch4: ../data/nc/ch4_lamongan.nc


## 8. Konversi NetCDF ke CSV

Setelah data NetCDF (`.nc`) tersedia, kita membaca tiap file menggunakan `xarray`,
mengubahnya menjadi `DataFrame` (satu baris per tanggal), kemudian menyimpannya sebagai
CSV ke folder `../data/csv/`. Format CSV inilah yang akan dipakai pada tahap Analisis
Data (Notebook 2).

In [7]:
def nc_to_csv(nc_path, label, csv_dir):
    """Membaca file NetCDF hasil ekstraksi dan mengonversinya menjadi CSV."""
    ds = xr.open_dataset(nc_path)
    df = ds.to_dataframe().reset_index()

    # Merapikan nama kolom: kolom waktu -> 'date', kolom nilai -> nama polutan
    rename_map = {}
    for col in df.columns:
        if "time" in col.lower() or "date" in col.lower():
            rename_map[col] = "date"
    df = df.rename(columns=rename_map)

    csv_path = os.path.join(csv_dir, f"{label}_lamongan.csv")
    df.to_csv(csv_path, index=False)
    print(f"[{label.upper()}] CSV disimpan -> {csv_path} ({len(df)} baris)")
    return csv_path


csv_paths = {}
for label, nc_path in nc_paths.items():
    csv_paths[label] = nc_to_csv(nc_path, label, CSV_DIR)

[NO2] CSV disimpan -> ../data/csv/no2_lamongan.csv (300 baris)
[CO] CSV disimpan -> ../data/csv/co_lamongan.csv (302 baris)
[SO2] CSV disimpan -> ../data/csv/so2_lamongan.csv (336 baris)
[CH4] CSV disimpan -> ../data/csv/ch4_lamongan.csv (57 baris)


## 9. Verifikasi Hasil Ekstraksi

Terakhir, kita cek daftar file yang berhasil dibuat di `../data/csv/`, lalu menampilkan
5 baris pertama dari salah satu CSV (NO₂) untuk memastikan data terlihat wajar sebelum
lanjut ke tahap analisis.

In [8]:
print("File CSV yang tersedia di", os.path.abspath(CSV_DIR), ":")
for f in sorted(os.listdir(CSV_DIR)):
    print(" -", f)

# Pratinjau salah satu hasil (NO2), jika berhasil dibuat
if "no2" in csv_paths:
    preview = pd.read_csv(csv_paths["no2"])
    display(preview.head())

File CSV yang tersedia di d:\Semester 5\Proyek sain data\PSD\data\csv :
 - ch4_lamongan.csv
 - co_lamongan.csv
 - no2_lamongan.csv
 - so2_lamongan.csv


,t,feature,NO2,lat,lon,feature_names
0,2025-08-24,0,0.000032,-7.075,112.3,feature_0
1,2025-08-25,0,0.000036,-7.075,112.3,feature_0
2,2025-08-26,0,0.000060,-7.075,112.3,feature_0
3,2025-08-27,0,0.000032,-7.075,112.3,feature_0
4,2025-08-28,0,0.000066,-7.075,112.3,feature_0
